In [ ]:
# ============================================================
# 1) INSTALLS
# ============================================================
!pip install -q ultralytics opencv-python pandas numpy

In [ ]:
# ============================================================
# 2) IMPORTS
# ============================================================
import os
import cv2
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

In [ ]:
# ============================================================
# 3) FIND YOUR VIDEO AND MODELS
# ============================================================
print("VIDEOS:")
!find /kaggle/input -type f \( -name "*.mp4" -o -name "*.mov" -o -name "*.avi" \)

print("\nMODELS:")
!find /kaggle/input /kaggle/working -type f \( -name "*.pt" -o -name "*.tflite" -o -name "*.onnx" \) 2>/dev/null
!find /kaggle/input /kaggle/working -type d -name "*ncnn*" 2>/dev/null

In [ ]:
# ============================================================
# 4) SET YOUR SINGLE VIDEO PATH HERE
# ============================================================

VIDEO_PATH = "/kaggle/input/YOUR_VIDEO_DATASET_NAME/YOUR_VIDEO.mp4"

OUTPUT_DIR = "/kaggle/working/video_model_tests"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Video:", VIDEO_PATH)
print("Output:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 5) AUTO-COLLECT ALL MODELS
# ============================================================

def collect_models():
    search_roots = ["/kaggle/input", "/kaggle/working"]
    model_paths = []

    for root in search_roots:
        for ext in ["*.pt", "*.tflite", "*.onnx"]:
            model_paths.extend(Path(root).rglob(ext))

        # NCNN export is usually a folder containing .param and .bin
        for d in Path(root).rglob("*ncnn*"):
            if d.is_dir():
                has_param = any(d.glob("*.param"))
                has_bin = any(d.glob("*.bin"))
                if has_param and has_bin:
                    model_paths.append(d)

    # Remove duplicates
    unique = []
    seen = set()
    for p in model_paths:
        p = str(p)
        if p not in seen:
            unique.append(p)
            seen.add(p)

    return unique


MODEL_PATHS = collect_models()

# Remove INT8 models if any were partially or fully exported
MODEL_PATHS = [
    p for p in MODEL_PATHS
    if "int8" not in str(p).lower()
]

# Optional: remove broken/empty files
MODEL_PATHS = [
    p for p in MODEL_PATHS
    if Path(p).is_dir() or Path(p).stat().st_size > 0
]

print("Found models after filtering:")
for i, p in enumerate(MODEL_PATHS):
    print(i, "->", p)

In [ ]:
# ============================================================
# 6) CLASS CONFIG
# ============================================================

CLASS_NAMES = {
    0: "ball",
    1: "car",
    2: "fallen-pins",
    3: "standing-pins"
}

BALL_ID = 0
CAR_ID = 1
FALLEN_ID = 2
STANDING_ID = 3

CONF_THRESH = {
    BALL_ID: 0.15,
    CAR_ID: 0.15,
    FALLEN_ID: 0.12,
    STANDING_ID: 0.12
}

IOU_MATCH_THRESHOLD = 0.10
CENTER_DISTANCE_THRESHOLD = 160
MISSING_KEEP_FRAMES = 25
FALL_CONFIRM_FRAMES = 3
MIN_TRACK_INIT_CONF = 0.12

PROCESS_EVERY_N_FRAMES = 1

In [ ]:
# ============================================================
# 7) HELPER FUNCTIONS
# ============================================================

def safe_name(path):
    p = Path(path)
    if p.is_dir():
        name = p.name
    else:
        name = p.stem

    name = name.replace(" ", "_")
    name = name.replace(".", "_")
    name = name.replace("-", "_")
    return name


def xyxy_to_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)


def box_area(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = box_area(boxA)
    areaB = box_area(boxB)
    union = areaA + areaB - inter

    if union <= 0:
        return 0.0

    return inter / union


def center_distance(boxA, boxB):
    return float(np.linalg.norm(xyxy_to_center(boxA) - xyxy_to_center(boxB)))


def draw_label(frame, text, x, y, color=(0, 255, 0), scale=0.7, thickness=2):
    cv2.putText(
        frame,
        text,
        (int(x), int(y)),
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        color,
        thickness,
        cv2.LINE_AA
    )


def draw_box(frame, box, color, label=None, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

    if label:
        draw_label(frame, label, x1, max(20, y1 - 8), color=color)

In [ ]:
# ============================================================
# 8) PIN TRACK CLASS
# ============================================================

class PinTrack:
    def __init__(self, track_id, box, cls_id, conf, frame_idx):
        self.track_id = track_id
        self.box = np.array(box, dtype=np.float32)
        self.last_seen_frame = frame_idx
        self.missing_frames = 0

        self.state = "standing" if cls_id == STANDING_ID else "fallen"
        self.last_cls_id = cls_id
        self.last_conf = conf

        self.fallen_candidate_count = 0
        self.confirmed_fallen = False
        self.fall_order = None
        self.fall_frame = None

        self.history = []

    def update(self, box, cls_id, conf, frame_idx):
        self.box = np.array(box, dtype=np.float32)
        self.last_seen_frame = frame_idx
        self.missing_frames = 0
        self.last_cls_id = cls_id
        self.last_conf = conf
        self.history.append((frame_idx, self.box.copy(), cls_id, conf))

        if cls_id == FALLEN_ID and not self.confirmed_fallen:
            self.fallen_candidate_count += 1
        elif cls_id == STANDING_ID and not self.confirmed_fallen:
            self.fallen_candidate_count = max(0, self.fallen_candidate_count - 1)

    def mark_missing(self):
        self.missing_frames += 1

    def should_confirm_fallen(self):
        return (not self.confirmed_fallen) and self.fallen_candidate_count >= FALL_CONFIRM_FRAMES

In [ ]:
# ============================================================
# 9) VIDEO SCORING PIPELINE
# ============================================================

def run_video_scoring(model_path, video_path, output_path, metrics_csv_path, model_name):
    print("\n" + "=" * 70)
    print("Testing model:", model_name)
    print("Path:", model_path)
    print("=" * 70)

    model = YOLO(model_path)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    elapsed_seconds = total_frames / fps if fps > 0 else 0

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    tracks = []
    next_track_id = 1
    next_fall_order = 1
    car_path = []

    frame_logs = []

    start_time = time.time()
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        display = frame.copy()

        if frame_idx % PROCESS_EVERY_N_FRAMES == 0:
            results = model.predict(
                source=frame,
                imgsz=640,
                conf=0.10,
                iou=0.45,
                verbose=False
            )[0]

            detections = []

            if results.boxes is not None:
                for b in results.boxes:
                    cls_id = int(b.cls[0])
                    conf = float(b.conf[0])
                    box = b.xyxy[0].cpu().numpy().astype(float)

                    if cls_id in CONF_THRESH and conf >= CONF_THRESH[cls_id]:
                        detections.append({
                            "box": box,
                            "cls_id": cls_id,
                            "conf": conf,
                            "name": CLASS_NAMES.get(cls_id, str(cls_id))
                        })

            pin_dets = [d for d in detections if d["cls_id"] in [FALLEN_ID, STANDING_ID]]
            car_dets = [d for d in detections if d["cls_id"] == CAR_ID]
            ball_dets = [d for d in detections if d["cls_id"] == BALL_ID]

            matched_dets = set()

            # Match existing tracks to detections
            for ti, track in enumerate(tracks):
                best_score = -999999
                best_di = None

                for di, det in enumerate(pin_dets):
                    if di in matched_dets:
                        continue

                    det_box = det["box"]
                    iou_score = iou(track.box, det_box)
                    dist = center_distance(track.box, det_box)

                    score = (iou_score * 2.0) - (dist / CENTER_DISTANCE_THRESHOLD)

                    if iou_score >= IOU_MATCH_THRESHOLD or dist <= CENTER_DISTANCE_THRESHOLD:
                        if score > best_score:
                            best_score = score
                            best_di = di

                if best_di is not None:
                    det = pin_dets[best_di]
                    track.update(det["box"], det["cls_id"], det["conf"], frame_idx)
                    matched_dets.add(best_di)
                else:
                    track.mark_missing()

            # Create new pin tracks
            for di, det in enumerate(pin_dets):
                if di in matched_dets:
                    continue

                if det["conf"] >= MIN_TRACK_INIT_CONF:
                    new_track = PinTrack(
                        track_id=next_track_id,
                        box=det["box"],
                        cls_id=det["cls_id"],
                        conf=det["conf"],
                        frame_idx=frame_idx
                    )
                    tracks.append(new_track)
                    next_track_id += 1

            # Confirm fallen pins
            for track in tracks:
                if track.should_confirm_fallen():
                    track.confirmed_fallen = True
                    track.state = "fallen"
                    track.fall_order = next_fall_order
                    track.fall_frame = frame_idx
                    next_fall_order += 1

            # Remove dead tracks
            tracks = [t for t in tracks if t.missing_frames <= MISSING_KEEP_FRAMES]

            # Track car path
            if car_dets:
                best_car = max(car_dets, key=lambda d: d["conf"])
                car_center = xyxy_to_center(best_car["box"])
                car_path.append((int(car_center[0]), int(car_center[1])))

            # Draw ball and car detections
            for det in ball_dets:
                draw_box(display, det["box"], (255, 255, 0), f"ball {det['conf']:.2f}")

            for det in car_dets:
                draw_box(display, det["box"], (255, 0, 255), f"car {det['conf']:.2f}")

        # Draw pin tracks
        for track in tracks:
            if track.confirmed_fallen:
                color = (0, 0, 255)
                label = f"FALL #{track.fall_order}"
            else:
                color = (0, 255, 0)
                label = f"PIN {track.track_id}"

            draw_box(display, track.box, color, label)

            if track.confirmed_fallen and track.fall_order is not None:
                cx, cy = xyxy_to_center(track.box)
                cv2.circle(display, (int(cx), int(cy)), 18, (0, 0, 255), -1)
                draw_label(
                    display,
                    str(track.fall_order),
                    int(cx) - 8,
                    int(cy) + 8,
                    color=(255, 255, 255),
                    scale=0.8,
                    thickness=2
                )

        # Draw car path
        for i in range(1, len(car_path)):
            cv2.line(display, car_path[i - 1], car_path[i], (255, 0, 255), 3)

        fallen_count = sum(1 for t in tracks if t.confirmed_fallen)

        # Top-left summary overlay
        cv2.rectangle(display, (10, 10), (470, 105), (0, 0, 0), -1)
        draw_label(display, f"Model: {model_name}", 20, 35, color=(255, 255, 255), scale=0.7)
        draw_label(display, f"Time: {frame_idx / fps:.2f}s / {elapsed_seconds:.2f}s", 20, 62, color=(255, 255, 255), scale=0.7)
        draw_label(display, f"Fallen pins: {fallen_count}", 20, 90, color=(255, 255, 255), scale=0.7)

        writer.write(display)

        frame_logs.append({
            "frame": frame_idx,
            "time_sec": frame_idx / fps if fps > 0 else 0,
            "active_tracks": len(tracks),
            "confirmed_fallen": fallen_count,
            "car_path_points": len(car_path)
        })

        frame_idx += 1

    cap.release()
    writer.release()

    runtime = time.time() - start_time

    final_tracks = []
    for t in tracks:
        final_tracks.append({
            "track_id": t.track_id,
            "confirmed_fallen": t.confirmed_fallen,
            "fall_order": t.fall_order,
            "fall_frame": t.fall_frame,
            "fall_time_sec": t.fall_frame / fps if t.fall_frame is not None and fps > 0 else None,
            "last_box": t.box.tolist(),
            "last_conf": float(t.last_conf)
        })

    summary = {
        "model_name": model_name,
        "model_path": str(model_path),
        "video_path": str(video_path),
        "output_path": str(output_path),
        "fps": float(fps),
        "width": int(width),
        "height": int(height),
        "total_frames": int(total_frames),
        "video_elapsed_seconds": float(elapsed_seconds),
        "processing_runtime_seconds": float(runtime),
        "processing_fps": float(total_frames / runtime) if runtime > 0 else 0,
        "final_fallen_count": int(sum(1 for t in tracks if t.confirmed_fallen)),
        "total_pin_tracks_remaining": int(len(tracks)),
        "car_path_points": int(len(car_path)),
        "final_tracks": final_tracks
    }

    pd.DataFrame(frame_logs).to_csv(metrics_csv_path, index=False)

    summary_path = metrics_csv_path.replace(".csv", "_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("Saved video:", output_path)
    print("Saved metrics:", metrics_csv_path)
    print("Saved summary:", summary_path)
    print("Final fallen count:", summary["final_fallen_count"])
    print("Processing FPS:", summary["processing_fps"])

    return summary

In [ ]:
# ============================================================
# 10) RUN SINGLE VIDEO ON ALL FOUND MODELS
# ============================================================

assert os.path.exists(VIDEO_PATH), f"Video path does not exist: {VIDEO_PATH}"
assert len(MODEL_PATHS) > 0, "No models found. Upload best.pt / tflite / onnx / ncnn model first."

all_summaries = []

for model_path in MODEL_PATHS:
    model_name = safe_name(model_path)

    output_video = f"{OUTPUT_DIR}/annotated_{model_name}.mp4"
    metrics_csv = f"{OUTPUT_DIR}/metrics_{model_name}.csv"

    try:
        summary = run_video_scoring(
            model_path=model_path,
            video_path=VIDEO_PATH,
            output_path=output_video,
            metrics_csv_path=metrics_csv,
            model_name=model_name
        )
        all_summaries.append(summary)

    except Exception as e:
        print("\nFAILED MODEL:", model_path)
        print("ERROR:", e)

In [ ]:
# ============================================================
# 11) CREATE MODEL COMPARISON CSV
# ============================================================

comparison = pd.DataFrame([
    {
        "model": s["model_name"],
        "model_path": s["model_path"],
        "output_video": s["output_path"],
        "video_duration_sec": s["video_elapsed_seconds"],
        "total_frames": s["total_frames"],
        "processing_runtime_sec": s["processing_runtime_seconds"],
        "processing_fps": s["processing_fps"],
        "fallen_count": s["final_fallen_count"],
        "pin_tracks_remaining": s["total_pin_tracks_remaining"],
        "car_path_points": s["car_path_points"]
    }
    for s in all_summaries
])

comparison_path = f"{OUTPUT_DIR}/model_comparison.csv"
comparison.to_csv(comparison_path, index=False)

comparison

In [ ]:
# ============================================================
# 12) ZIP ALL OUTPUTS FOR DOWNLOAD
# ============================================================

!zip -r /kaggle/working/video_model_tests.zip /kaggle/working/video_model_tests

print("Download this file:")
print("/kaggle/working/video_model_tests.zip")